# Market Technical Indicators

## Problem Definition

**Question.** Are the stored AAPL 2025 technical indicators complete and aligned to completed dollar bars?

**Role in the workflow.** Validate the event-start technical feature block used by the primary model.

**Inputs.** Local dollar bars and the local technical-feature Parquet.

**Outputs.** A feature-family coverage summary and the validated technical path.

**Why this method.** Using completed-bar indicators permits backward/as-of event alignment without external calls or future bars.

**Assumptions.** Indicator values are descriptive transformations, can be redundant, and are not individually selected using holdout performance.

**Handoff.** The technical feature path to `event_labeling.ipynb`.


## Real Data Check and Preprocessing Boundary

The notebook checks feature alignment and completeness only. Model-specific comparison and importance are deferred to the primary and meta notebooks.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
technical_path = feature_dir / f"aapl_dollar_bar_technical_{period}.parquet"

dollar_bars = pd.read_parquet(dollar_path)[["start", "end", "symbol"]]
technical = pd.read_parquet(technical_path)
identifier_columns = ["start", "end", "symbol"]
feature_columns = [column for column in technical.columns if column not in identifier_columns]

assert len(feature_columns) == 51
assert technical["symbol"].eq("AAPL").all()
assert technical["end"].equals(dollar_bars["end"])
missing_by_feature = technical[feature_columns].isna().sum()
assert missing_by_feature.lt(len(technical)).all()

families = pd.Series(
    {
        "breadth": 6,
        "momentum": 18,
        "overlap": 15,
        "volatility": 12,
    },
    name="feature_count",
)
assert families.sum() == len(feature_columns)

display(families.to_frame())
display(missing_by_feature[missing_by_feature.gt(0)].sort_values(ascending=False).to_frame("missing_rows"))
display(technical[identifier_columns + feature_columns[:5]].head())


## Results, Limitations, and Handoff

Technical indicators share common price inputs and can substitute for one another. That redundancy is measured with MDI, MDA, SFI, and orthogonal analysis inside each fitted model workflow.

The next notebook receives the 51 aligned technical features. No conclusion in this notebook is evidence of live-trading profitability.
